In [2]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth.csv") #-data.csv")

In [3]:
df_ground_truth.head()

,question,document
0,I just found this course late — can I still jo...,74eb249bbf
1,Am I allowed to start the course after it has ...,74eb249bbf
2,"If I join late, can I still get the certificat...",74eb249bbf
3,What do I need to do to be eligible for the co...,74eb249bbf
4,Is it okay to take the course after it started...,74eb249bbf


In [4]:
ground_truth = df_ground_truth.to_dict(orient="records")

In [5]:
ground_truth[10]

{'question': 'Where can I watch the Office Hours or live workshop stream if I’m a student?',
 'document': '489dd1c9d9'}

In [6]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [7]:
boost = {'question': 3.0}

index.search(
    ground_truth[0]["question"],
    num_results=5,
    boost_dict=boost
)

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'a9353fadfe',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The homework submission form is still open even though the deadline has passed — can I still submit?',
  'answer': "Yes. As long as the submission form is still open, you can submit your answers, even if the listed deadline has already passed. You can no longer submit only after the form has been closed — so while it's still open, go ahead and submit."},
 {'id': '9f689c185f',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I missed the first homework - can I still get a certificate?',
  'answer': 'Yes, you need to pass the Capstone proj

In [8]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [9]:
q = ground_truth[0]
q

{'question': 'I just found this course late — can I still join now?',
 'document': '74eb249bbf'}

In [10]:
doc_id = q['document']
doc_id

'74eb249bbf'

In [11]:
results = text_search(q['question'])

In [12]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
a9353fadfe == 74eb249bbf: False
9f689c185f == 74eb249bbf: False
85384a18e5 == 74eb249bbf: False
977bf7786c == 74eb249bbf: False


In [13]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

In [14]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [15]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)

I just found this course late — can I still join now?


[1, 0, 0, 0, 0]

In [18]:
q = ground_truth[12]
print(q["question"])
compute_relevance_text(q)

How do I submit questions during a live session if I’m watching on YouTube?


[0, 1, 0, 0, 0]

In [19]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [20]:
relevance = compute_relevance_total_text(ground_truth)

  0%|          | 0/515 [00:00<?, ?it/s]

In [21]:
relevance[:15]

[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [24]:
sample = relevance[:15]
sample

[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [25]:
12/15

0.8

In [39]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [40]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [26]:
cnt = 0

for line in sample:
    if 1 in line:
        cnt = cnt + 1

cnt / len(sample)

0.8

In [27]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [28]:
hit_rate(relevance)

0.8660194174757282

In [29]:
total_score = 0.0

for line in sample:
    for rank in range(len(line)):
        if line[rank] == 1:
            score = 1 / (rank + 1)
            total_score = total_score + score
            break

total_score / len(sample)

0.6888888888888888

In [30]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break

    return total_score / len(relevance)

In [31]:
mrr(relevance)

0.75378640776699

In [41]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [42]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [43]:
evaluate(ground_truth, text_search)

  0%|          | 0/515 [00:00<?, ?it/s]

{'hit_rate': 0.8660194174757282, 'mrr': 0.75378640776699}

In [44]:
def text_search_v2(query):
    boost_dict = {"question": 2.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [45]:
evaluate(ground_truth, text_search_v2)

  0%|          | 0/515 [00:00<?, ?it/s]

{'hit_rate': 0.887378640776699, 'mrr': 0.7795469255663429}

In [46]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [47]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

  0%|          | 0/515 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.9029126213592233, 'mrr': 0.813009708737864}


  0%|          | 0/515 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.9184466019417475, 'mrr': 0.8155339805825239}


  0%|          | 0/515 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.8660194174757282, 'mrr': 0.75378640776699}


  0%|          | 0/515 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.8349514563106796, 'mrr': 0.7266019417475729}


  0%|          | 0/515 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.8097087378640777, 'mrr': 0.6859870550161812}


In [48]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [49]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(f"Evaluating question_boost={question_boost}, answer_boost={answer_boost}, section_boost={section_boost}...")
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/515 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/515 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/515 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/515 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/515 [00:00<?, ?it/s]

In [50]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
4,1.0,2.0,0.2,0.974757,0.888608
18,2.0,4.0,0.1,0.974757,0.886667
3,1.0,2.0,0.1,0.972816,0.885728
19,2.0,4.0,0.2,0.972816,0.885728
35,5.0,10.0,0.5,0.972816,0.885728
34,5.0,10.0,0.2,0.972816,0.885307
20,2.0,4.0,0.5,0.974757,0.885113
33,5.0,10.0,0.1,0.972816,0.882686
7,1.0,4.0,0.2,0.972816,0.881456
6,1.0,4.0,0.1,0.970874,0.877735
